# 🚢 Training Pipeline: Baltic Capesize Freight Index (CI) Model
### Smart India Hackathon (SIH 2024) — Vessel Chartering & Bulk Procurement
This interactive notebook trains, evaluates, and forecasts the **Baltic Capesize Index (CI)** for 100,000–220,000 DWT heavy raw material bulk carriers (iron ore, coking coal).


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('dataset_used_capesize_bdi.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print("Capesize Dataset Loaded! Shape:", df.shape)


### 1. Feature Engineering & Split


In [ ]:
for lag in [1, 2, 3, 5, 7, 14, 21, 30]:
    df[f'CI_lag_{lag}'] = df['CI'].shift(lag)

for w in [7, 14, 30]:
    df[f'CI_roll_mean_{w}'] = df['CI'].shift(1).rolling(w).mean()
    df[f'CI_roll_std_{w}']  = df['CI'].shift(1).rolling(w).std()

df['dayofweek'] = df['Date'].dt.dayofweek
df['month'] = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter

data = df.dropna().reset_index(drop=True)
feature_cols = [c for c in data.columns if c not in ['Date', 'CI']]

split_idx = int(len(data) * 0.8)
train_df, test_df = data.iloc[:split_idx], data.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['CI']
X_test, y_test = test_df[feature_cols], test_df['CI']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 2. Model Training & Metrics


In [ ]:
models = {
    'Ridge': Ridge(alpha=10.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42, verbosity=0)
}

results, predictions = [], {'Date': test_df['Date'].values, 'Actual': y_test.values}

for name, model in models.items():
    if name == 'Ridge':
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    
    predictions[name] = preds
    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE (%)': mape})

results_df = pd.DataFrame(results).sort_values('MAE')
print(results_df.to_string(index=False))


### 3. Save Model


In [ ]:
best_model = models['GradientBoosting']
joblib.dump(best_model, 'CI_best_model.joblib')
pd.DataFrame(predictions).to_csv('test_predictions.csv', index=False)
print("Capesize best model saved to 'CI_best_model.joblib'!")
